### Setup

In [1]:
# install dependencies for virtual env (requirements.txt is in the repo root)
!pip install -r ../requirements.txt -qqq

In [2]:
# import librariess
import sys
import torch

#let this notebook import our helpers from src/ in the repo root
sys.path.append("..")
from src.model import load_model, chat

In [3]:
#declaring which layer to inspect
LAYER = 18

#download and load qwen's tokenizer and model on the gpu if there is one (code lives in src/model.py)
tok, model, device = load_model("Qwen/Qwen3-0.6B")

#check model config: # of hidden layers and dimensionality of each layer
print(model.config.num_hidden_layers, model.config.hidden_size)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


W1116 12:45:39.081000 66804 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

28 1024


### Basics

In [4]:
text = "Rent was due in two days and Maria refreshed the bank page again."

#tokenize the above input, maps each token to an ID and returns a tensor of those IDs. 
#these IDs are then moved to the same device the model lives on 
#here, im running this on my mac, so it moves it to the MPS device the model is on. 
ids = tok(text, return_tensors="pt").to(device)

#return the shape of ID tensor created above
print(ids.input_ids.shape)

#return the tokens corresponding to the IDs
print(tok.convert_ids_to_tokens(ids.input_ids[0]))

torch.Size([1, 14])
['Rent', 'Ġwas', 'Ġdue', 'Ġin', 'Ġtwo', 'Ġdays', 'Ġand', 'ĠMaria', 'Ġrefreshed', 'Ġthe', 'Ġbank', 'Ġpage', 'Ġagain', '.']


### Extract Internal States

In [5]:
#disable gradient computation graph, and feed the token IDs into the model.
with torch.no_grad():
    out = model(**ids, output_hidden_states=True)

#print the number of hidden layers (28+1)
print(len(out.hidden_states))

#print the shape of residual stream of vectors @ the 18th layer
#14 vectors, 1 for each token, each of size 1024 dimensions. 
print(out.hidden_states[LAYER].shape)

29
torch.Size([1, 14, 1024])


### Generate Text from the Model

In [6]:
#chat() lives in src/model.py; it takes the model and tokenizer as inputs
print(chat(model, tok, "Write one sentence about a desperate person."))

A desperate person, trapped in a moment of crisis, searches desperately for hope.


whats next? actual probing/steering! 